# H1 · Displacement Equation: Two Set-ups and Newton–Raphson
**Ship and Ocean Systems Design 2026 · Homework H1 · optional, not graded · individual**

Extends **Exercise 1.5** (Lecture 1.5, slide 7): questions 7 and 8 of the displacement exercise, on **the reference ship**, the 37,600 t oil/chemical tanker (owner's brief: 0.6 The Reference Ship, slide 2). Practice for D3. Papanikolaou (2026) §2.13 and Appendix C.

Questions 1–6 were done in class, individually; their worked answers are in **Exercise Answers – Week 1** and the Exercise 1.5 deck. This notebook rebuilds the model in a few lines, then asks:
* **Q7** Does it matter how the iteration is set up? (Form A against Form B)
* **Q8** Solve the same equation with Newton–Raphson and compare.

Run each cell, then answer the questions in the markdown cells in your own words.

In [ ]:
import numpy as np
import pandas as pd

# Data of the exercise statement (Lecture 1.5, slide 7)
DWT, V, C_adm = 37_600.0, 14.5, 450.0           # t, kn, -  (owner's brief; Admiralty constant, course assumption)
rho, CB = 1.025, 0.757                          # t/m3; Horn's CB from Exercise 1.4
LB, BT, DT = 5.92, 2.69, 1.36                   # L/B, B/T, D/T from Exercise 1.4 (tanker fits, Appendix A)
K, c_o, k_m = 0.032, 0.025, 0.10                # Watson K (tankers, Table 2.24); outfit coeff.; machinery [t/kW]

def dims(D):                                    # L, B, T, depth for a displacement D
    L = (D * LB**2 * BT / (rho * CB)) ** (1/3)
    B = L / LB; T = B / BT
    return L, B, T, T * DT

def W_st(D):                                    # Watson & Gilfillan, simplified
    L, B, T, Dp = dims(D)
    E = L*(B + T) + 0.85*L*(Dp - T)
    return K * E**1.36

W_o = lambda D: c_o * D
W_m = lambda D: k_m * D**(2/3) * V**3 / C_adm
g   = lambda D: DWT + W_st(D) + W_o(D) + W_m(D)                      # Form A: Delta = g(Delta)
dg  = lambda D: c_o + (1.36*2/3)*W_st(D)/D + (2/3)*W_m(D)/D          # g'(Delta)

# Question 6 again: Form A from Delta0 = DWT
D = DWT
for k in range(50):
    D_new = g(D)
    if abs(D_new - D) < 1e-9: break
    D = D_new
D_star = D
L, B, T, Dp = dims(D_star)
print(f"Delta* = {D_star:.1f} t   L x B x T x D = {L:.1f} x {B:.2f} x {T:.2f} x {Dp:.2f} m")
print(f"W_st = {W_st(D_star):.1f} t, W_o = {W_o(D_star):.1f} t, W_m = {W_m(D_star):.1f} t, g'(Delta*) = {dg(D_star):.4f}")

## Q7 · Does it matter how the iteration is set up?
In the equilibrium equation Δ appears four times: in the buoyancy and inside each of the three lightship groups. **Form A** (Question 4) takes the buoyancy as the new value: add up the weights of the current ship, and that total is the next displacement.

**Form B** starts from the machinery:
1. take Δₖ and subtract deadweight, steel and outfit: what is left is the weight available for machinery;
2. turn it into power: P_B = W_m / k_m;
3. find the displacement that needs exactly that power, from the Admiralty formula: Δₖ₊₁ = (P_B · C_adm / V³)^(3/2).

The convergence of Form B is governed by h′(Δ) = (1 − c_o − dW_st/dΔ) / (dW_m/dΔ).

In [ ]:
def h(D):                                       # Form B
    W_m_avail = D - DWT - W_st(D) - W_o(D)
    P_B = W_m_avail / k_m
    return (P_B * C_adm / V**3) ** 1.5 if P_B > 0 else float("nan")

dWst = lambda D: (1.36*2/3) * W_st(D) / D
dWm  = lambda D: (2/3) * W_m(D) / D
dh   = lambda D: (1 - c_o - dWst(D)) / dWm(D)

print(f"(a) h(Delta*) = {h(D_star):.1f} t   (Delta* = {D_star:.1f} t): both forms share the solution")
print(f"(b) h'(Delta*) = {dh(D_star):.2f}   against g'(Delta*) = {dg(D_star):.3f}")

rows = []
DA = DB = 46_010.0
for k in range(4):
    rows.append(dict(k=k, form_A=DA, error_A=DA - D_star, form_B=DB, error_B=DB - D_star))
    DA, DB = g(DA), h(DB)
print("(c) three iterations from 46,010 t  (NaN: no weight left for machinery, Form B has left the physical range)")
pd.DataFrame(rows).set_index("k").round(2)

**(d) Explain in your own words** why one set-up works and the other fails. *Hint:* in Form A, a change of Δ moves the lightship by g′ ≈ 0.16 of it; in Form B, a small error in the weight left for machinery is divided by dW_m/dΔ, which is small. Which number multiplies the error at each pass?

*Your answer:*

## Q8 · Newton–Raphson
Rewrite the equilibrium as the excess buoyancy f(Δ) = Δ − g(Δ), which is zero at equilibrium, negative when the ship is too small to carry its own weight. Newton–Raphson follows the tangent: Δₖ₊₁ = Δₖ − f(Δₖ) / f′(Δₖ), with f′ = 1 − g′.

In [ ]:
f  = lambda D: D - g(D)
fp = lambda D: 1 - dg(D)

# (a) f' is never zero for Delta >= DWT, because g' <= g'(DWT) < 1
print(f"(a) g'(DWT) = {dg(DWT):.4f}  ->  f' >= {1 - dg(DWT):.4f} for every Delta >= DWT")

# (b) convexity: f'' = -g'' > 0 (numerical check over the range)
Ds = np.linspace(DWT, 2*DWT, 50)
fpp = [(fp(x + 1) - fp(x - 1)) / 2 for x in Ds]
print(f"(b) f'' between {min(fpp):.2e} and {max(fpp):.2e} 1/t: positive, so f is convex")

# (c) iterate to a change below 1 t, as in the exercise (Question 8)
rows, D = [], DWT
for k in range(10):
    step = -f(D) / fp(D)
    rows.append(dict(k=k, Delta=D, f=f(D), f_prime=fp(D), step=step))
    D += step
    if abs(step) < 1.0: break
rows.append(dict(k=k+1, Delta=D, f=f(D), f_prime=fp(D), step=float("nan")))
pd.DataFrame(rows).set_index("k")

In [ ]:
# (d) compare with the fixed point of Question 6: error per step
fixed, newton = [], []
D = DWT
for k in range(8):
    fixed.append(abs(D - D_star)); D = g(D)
D = DWT
for k in range(4):
    newton.append(abs(D - D_star)); D = D - f(D) / fp(D)
err = pd.DataFrame({"fixed point |e_k| [t]": pd.Series(fixed), "Newton |e_k| [t]": pd.Series(newton)})
print(err.to_string(float_format=lambda x: f"{x:.2e}"))

**(d) Comment** on the number of iterations, the rate of convergence (linear against quadratic) and the extra work each Newton step needs. When would you use each method inside a design loop that solves this equation thousands of times?

*Your answer:*

---
Then take **Quiz 1**, the week's self-assessment (link from the lecturer).

---
## Optional · use this notebook for your project
The Ship Design Project is group work done outside class; this section is not part of the homework. To try the same solution on your group's own ship, change the data cell (deadweight, speed, ratios and coefficients from your D2 reference ships) and run the notebook again. Check |g′| < 1 before you trust the fixed point, and note any weight group your ship has that this model leaves out (batteries or LNG tanks, for example).